### Data Preparation for the Classifier

In [17]:
#imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [18]:
#data extraction
dataset=pd.DataFrame()
for type in ["Normal","DoS","DDoS","Data_Exfiltration","OS_Fingerprint","Service_Scan","Keylogging"] :
    data_type=pd.DataFrame()
    for i in range(1,75) :
        data=pd.read_csv(f"/teamspace/studios/this_studio/bot-iot/data_{i}.csv")
        if type in ["Normal","DoS","DDoS"] :
           data=data[data["category"]==type]
           data_type=pd.concat([data_type,data],axis=0,ignore_index=True)
           if len(data_type)>=9547 :
               break
        else :
           data=data[data["subcategory "]==type]
           data_type=pd.concat([data_type,data],axis=0,ignore_index=True)
           if len(data_type)>=9547 :
               break
    if type=="Data_Exfiltration" :
        dataset=pd.concat([dataset,data_type],axis=0,ignore_index=True)
    else :
        dataset=pd.concat([dataset,data_type.sample(frac=1).iloc[:9547]],axis=0,ignore_index=True)
    print(f"{type} done.")
    

Normal done.
DoS done.
DDoS done.
Data_Exfiltration done.
OS_Fingerprint done.
Service_Scan done.
Keylogging done.


In [19]:
dataset.category.value_counts()

category
Reconnaissance    19094
DDoS               9547
DoS                9547
Normal             9543
Theft              1587
Name: count, dtype: int64

In [20]:
dataset=dataset.sample(frac=1)
dataset.head()

,pkSeqID,stime,flgs,proto,saddr,sport,daddr,dport,pkts,bytes,...,spkts,dpkts,sbytes,dbytes,rate,srate,drate,attack,category,subcategory
37247,1688045,1.526963e+09,e,tcp,192.168.100.149,52760,192.168.100.5,32770,2,120,...,1,1,60,60,242.248077,0.000000,0.000000,1,Reconnaissance,OS_Fingerprint
13201,1889829,1.528081e+09,e s,tcp,192.168.100.147,23639,192.168.100.7,80,4,616,...,4,0,616,0,0.094425,0.094425,0.000000,1,DoS,TCP
30334,1827052,1.526982e+09,e,tcp,192.168.100.147,50133,192.168.100.7,1100,2,120,...,1,1,60,60,4464.285645,0.000000,0.000000,1,Reconnaissance,OS_Fingerprint
12636,1878485,1.528081e+09,e s,tcp,192.168.100.148,5838.0,192.168.100.6,80.0,7,890,...,5,2,770,120,0.191534,0.127863,0.048721,1,DoS,TCP
31374,1826195,1.526982e+09,e,tcp,192.168.100.149,48070,192.168.100.7,1434,2,120,...,1,1,60,60,1766.784424,0.000000,0.000000,1,Reconnaissance,OS_Fingerprint


#### Data cleaning

In [2]:
import pandas as pd
import numpy as np

In [47]:
data=pd.read_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/bot_iot_dataset.csv")
data.head()

,pkSeqID,stime,flgs,proto,saddr,sport,daddr,dport,pkts,bytes,...,spkts,dpkts,sbytes,dbytes,rate,srate,drate,attack,category,subcategory
0,1501064,1.526877e+09,e,tcp,192.168.100.148,64874,192.168.100.7,50002,2,120,...,1,1,60,60,2890.173584,0.000000,0.000000,1,Reconnaissance,OS_Fingerprint
1,34849863,1.528103e+09,e,tcp,192.168.100.148,42496,192.168.100.3,80,8,1484,...,5,3,784,700,0.967206,0.552689,0.323917,1,DDoS,HTTP
2,1111500,1.526955e+09,e,udp,192.168.100.3,27658,205.251.195.213,53,2,401,...,1,1,100,301,4.237557,0.000000,0.000000,0,Normal,Normal
3,1593051,1.526895e+09,e,udp,192.168.100.7,45454,192.168.159.152,53,2,176,...,2,0,176,0,0.198096,0.198096,0.000000,0,Normal,Normal
4,1842119,1.528089e+09,e,tcp,192.168.100.149,37742.0,192.168.100.3,80.0,10,1618,...,6,4,852,766,0.339983,0.188879,0.266181,1,DoS,HTTP


In [21]:
data=dataset.copy()

In [36]:
def transform(data):
    # Precompute grouped sums for bytes, pkts, and rate
    saddr_bytes_sum = data.groupby("saddr")["bytes"].sum()
    daddr_bytes_sum = data.groupby("daddr")["bytes"].sum()
    saddr_pkts_sum = data.groupby("saddr")["pkts"].sum()
    daddr_pkts_sum = data.groupby("daddr")["pkts"].sum()
    proto_pkts_sum = data.groupby("proto")["pkts"].sum()
    dport_pkts_sum = data.groupby("dport")["pkts"].sum()
    
    proto_saddr_rate_mean = data.groupby(["proto", "saddr"])["rate"].mean()
    proto_daddr_rate_mean = data.groupby(["proto", "daddr"])["rate"].mean()
    proto_sport_rate_mean = data.groupby(["proto", "sport"])["rate"].mean()
    proto_dport_rate_mean = data.groupby(["proto", "dport"])["rate"].mean()

    # Precompute counts for inbound connections
    daddr_inbound_count = data["daddr"].value_counts()
    
    # Precompute grouped sums for pkts based on proto and state
    pkts_state_proto_daddr = data.groupby(["proto", "state", "daddr"])["pkts"].sum()
    pkts_state_proto_saddr = data.groupby(["proto", "state", "saddr"])["pkts"].sum()
    
    # Add tnbpscip
    data["tnbpsrcip"] = data["saddr"].map(saddr_bytes_sum).fillna(0)
    print("tnbpscip done.")
    
    # Add tnbpdstip
    data["tnbpdstip"] = data["daddr"].map(daddr_bytes_sum).fillna(0)
    print("tnbpdstip done.")
    
    # Add tnp_psrcip
    data["tnp_psrcip"] = data["saddr"].map(saddr_pkts_sum).fillna(0)
    print("tnp_psrcip done.")
    
    # Add tnp_pdstip
    data["tnp_pdstip"] = data["daddr"].map(daddr_pkts_sum).fillna(0)
    print("tnp_pdstip done.")
    
    # Add tnp_perproto
    data["tnp_perproto"] = data["proto"].map(proto_pkts_sum).fillna(0)
    print("tnp_perproto done.")
    
    # Add tnp_per_dport
    data["tnp_per_dport"] = data["dport"].map(dport_pkts_sum).fillna(0)
    print("tnp_per_dport done.")
    
    # Add ar_p_proto_p_srcip
    data["ar_p_proto_p_srcip"] = data.apply(lambda x: proto_saddr_rate_mean.get((x["proto"], x["saddr"]), np.nan), axis=1)
    print("ar_p_proto_p_srcip done.")
    
    # Add ar_p_proto_p_dstip
    data["ar_p_proto_p_dstip"] = data.apply(lambda x: proto_daddr_rate_mean.get((x["proto"], x["daddr"]), np.nan), axis=1)
    print("ar_p_proto_p_dstip done.")
    
    # Add n_in_conn_p_srcip
    data["n_in_conn_p_srcip"] = data["saddr"].map(daddr_inbound_count).fillna(0)
    print("n_in_conn_p_srcip done.")
    
    # Add n_in_conn_p_dstip
    data["n_in_conn_p_dstip"] = data["daddr"].map(daddr_inbound_count).fillna(0)
    print("n_in_conn_p_dstip done.")
    
    # Add ar_p_proto_p_sport
    data["ar_p_proto_p_sport"] = data.apply(lambda x: proto_sport_rate_mean.get((x["proto"], x["sport"]), np.nan), axis=1)
    print("ar_p_proto_p_sport done.")
    
    # Add ar_p_proto_p_dport
    data["ar_p_proto_p_dport"] = data.apply(lambda x: proto_dport_rate_mean.get((x["proto"], x["dport"]), np.nan), axis=1)
    print("ar_p_proto_p_dport done.")
    
    # Add pkts_p_state_p_protocol_p_destip
    data["pkts_p_state_p_protocol_p_destip"] = data.apply(lambda x: pkts_state_proto_daddr.get((x["proto"], x["state"], x["daddr"]), 0), axis=1)
    print("pkts_p_state_p_protocol_p_destip done.")
    
    # Add pkts_p_state_p_protocol_p_srcip
    data["pkts_p_state_p_protocol_p_srcip"] = data.apply(lambda x: pkts_state_proto_saddr.get((x["proto"], x["state"], x["saddr"]), 0), axis=1)
    print("pkts_p_state_p_protocol_p_srcip done.")

In [37]:
transform(data)

tnbpscip done.
tnbpdstip done.
tnp_psrcip done.
tnp_pdstip done.
tnp_perproto done.
tnp_per_dport done.
ar_p_proto_p_srcip done.
ar_p_proto_p_dstip done.
n_in_conn_p_srcip done.
n_in_conn_p_dstip done.
ar_p_proto_p_sport done.
ar_p_proto_p_dport done.
pkts_p_state_p_protocol_p_destip done.
pkts_p_state_p_protocol_p_srcip done.


In [38]:
metadata={'pkSeqID': 'Row Identifier',
 'stime': 'Record start time',
 'flgs': 'Flow state flags seen in transactions',
 'proto': 'Textual representation of transaction protocols present in network flow',
 'saddr': 'Source IP address',
 'sport': 'Source port number',
 'daddr': 'Destination IP address',
 'dport': 'Destination port number',
 'pkts': 'Total count of packets in transaction',
 'bytes': 'Totan number of bytes in transaction',
 'state': 'Transaction state',
 'ltime': 'Record last time',
 'seq': 'Argus sequence number',
 'dur': 'Record total duration',
 'mean': 'Average duration of aggregated records',
 'stddev': 'Standard deviation of aggregated records',
 'sum': 'Total duration of aggregated records',
 'min': 'Minimum duration of aggregated records',
 'max': 'Maximum duration of aggregated records',
 'spkts': 'Source-to-destination packet count',
 'dpkts': 'Destination-to-source packet count',
 'sbytes': 'Source-to-destination byte count',
 'dbytes': 'Destination-to-source byte count',
 'rate': 'Total packets per second in transaction',
 'srate': 'Source-to-destination packets per second',
 'drate': 'Destination-to-source packets per second',
 'tnbpsrcip': 'Total Number of bytes per source IP',
 'tnbpdstip': 'Total Number of bytes per Destination IP.',
 'tnp_psrcip': 'Total Number of packets per source IP.',
 'tnp_pdstip': 'Total Number of packets per Destination IP.',
 'tnp_perproto': 'Total Number of packets per protocol.',
 'tnp_per_dport': 'Total Number of packets per dport',
 'ar_p_proto_p_srcip': 'Average rate per protocol per Source IP. (calculated by pkts/dur)',
 'ar_p_proto_p_dstip': 'Average rate per protocol per Destination IP.',
 'n_in_conn_p_srcip': 'Number of inbound connections per source IP.',
 'n_in_conn_p_dstip': 'Number of inbound connections per destination IP.',
 'ar_p_proto_p_sport': 'Average rate per protocol per sport',
 'ar_p_proto_p_dport': 'Average rate per protocol per dport',
 'pkts_p_state_p_protocol_p_destip': 'Number of packets grouped by state of flows and protocols per destination IP.',
 'pkts_p_state_p_protocol_p_srcip': 'Number of packets grouped by state of flows and protocols per source IP.',
 'attack': 'Class label: 0 for Normal traffic, 1 for Attack Traffic',
 'category': 'Traffic category',
 'subcategory ': 'Traffic subcategory'}

In [22]:
metadata={'pkSeqID': 'Row Identifier',
 'stime': 'Record start time',
 'flgs': 'Flow state flags seen in transactions',
 'proto': 'Textual representation of transaction protocols present in network flow',
 'saddr': 'Source IP address',
 'sport': 'Source port number',
 'daddr': 'Destination IP address',
 'dport': 'Destination port number',
 'pkts': 'Total count of packets in transaction',
 'bytes': 'Totan number of bytes in transaction',
 'state': 'Transaction state',
 'ltime': 'Record last time',
 'seq': 'Argus sequence number',
 'dur': 'Record total duration',
 'mean': 'Average duration of aggregated records',
 'stddev': 'Standard deviation of aggregated records',
 'sum': 'Total duration of aggregated records',
 'min': 'Minimum duration of aggregated records',
 'max': 'Maximum duration of aggregated records',
 'spkts': 'Source-to-destination packet count',
 'dpkts': 'Destination-to-source packet count',
 'sbytes': 'Source-to-destination byte count',
 'dbytes': 'Destination-to-source byte count',
 'rate': 'Total packets per second in transaction',
 'srate': 'Source-to-destination packets per second',
 'drate': 'Destination-to-source packets per second',
 'attack': 'Class label: 0 for Normal traffic, 1 for Attack Traffic',
 'category': 'Traffic category',
 'subcategory ': 'Traffic subcategory'}

In [39]:
data=data[list(metadata.keys())]
data.head()

,pkSeqID,stime,flgs,proto,saddr,sport,daddr,dport,pkts,bytes,...,ar_p_proto_p_dstip,n_in_conn_p_srcip,n_in_conn_p_dstip,ar_p_proto_p_sport,ar_p_proto_p_dport,pkts_p_state_p_protocol_p_destip,pkts_p_state_p_protocol_p_srcip,attack,category,subcategory
37247,1688045,1.526963e+09,e,tcp,192.168.100.149,52760,192.168.100.5,32770,2,120,...,939.965691,641.0,5208,564.567137,10430.133552,6482,23345,1,Reconnaissance,OS_Fingerprint
13201,1889829,1.528081e+09,e s,tcp,192.168.100.147,23639,192.168.100.7,80,4,616,...,1254.250649,120.0,6669,0.094425,100.862697,12994,15429,1,DoS,TCP
30334,1827052,1.526982e+09,e,tcp,192.168.100.147,50133,192.168.100.7,1100,2,120,...,1254.250649,120.0,6669,4368.510051,18097.726590,13229,23301,1,Reconnaissance,OS_Fingerprint
12636,1878485,1.528081e+09,e s,tcp,192.168.100.148,5838,192.168.100.6,80,7,890,...,9901.705447,146.0,5063,0.191534,100.862697,14111,30428,1,DoS,TCP
31374,1826195,1.526982e+09,e,tcp,192.168.100.149,48070,192.168.100.7,1434,2,120,...,1254.250649,641.0,6669,1856.201331,5937.703676,13229,23345,1,Reconnaissance,OS_Fingerprint


In [25]:
data["sport"].fillna(value="80",inplace=True)
data["dport"].fillna(value="53",inplace=True)

In [24]:
data["sport"]=data["sport"].astype(str)
data["dport"]=data["dport"].astype(str)

In [29]:
data["sport"]=data.sport.apply(lambda x : "80" if x=="nan" else x)
data["dport"]=data.dport.apply(lambda x : "53" if x=="nan" else x)

In [31]:
data.sport.value_counts().sort_values(ascending=False).head(20)

sport
80         962
80.0       695
57184      659
57044      628
54114      622
59460      613
56304      588
40081      583
47422      542
42640      509
40850      480
65233      363
41607      304
0x0303     272
65233.0    269
365.0      187
49731      187
0.0        187
3456.0     170
41307.0    157
Name: count, dtype: int64

In [32]:
def treat_sport(x) :
    if "." in x :
        return x.split(".")[0]
    else : 
        return x
data["sport"]=data["sport"].apply(treat_sport)
data["dport"]=data["dport"].apply(treat_sport)

In [34]:
data.sport.value_counts().sort_values(ascending=False).head(20)

sport
80        1657
57184      659
40081      633
65233      632
59460      630
57044      628
40850      628
54114      622
47422      609
56304      596
42640      529
41607      348
0          310
365        310
3456       274
0x0303     272
41307      268
8080       218
49731      207
5353       169
Name: count, dtype: int64

In [73]:
data[data["category"]=="DoS"].saddr.unique()

array(['192.168.100.149', '192.168.100.148', '192.168.100.150',
       '192.168.100.147', '192.168.100.3', '192.168.100.7',
       '192.168.100.6'], dtype=object)

In [40]:
for type in ["saddr","daddr"] :    
    for addr in ['192.168.100.149', '192.168.100.148', '192.168.100.150',
                '192.168.100.147', '192.168.100.3', '192.168.100.7',
                '192.168.100.6','192.168.100.5','192.168.100.4',"192.168.100.1"] :
            data[f"{type}_{addr}"]=data[type].apply(lambda x : 1 if x==addr else 0)
    data[f"{type}_private"]=data[type].apply(lambda x : 1 if "192.168.100" in x else 0)   
    data[f"{type}_external"]=data[type].apply(lambda x : 0 if "192.168.100" in x else 1)

In [76]:
data.columns

Index(['pkSeqID', 'stime', 'flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport',
       'pkts', 'bytes', 'state', 'ltime', 'seq', 'dur', 'mean', 'stddev',
       'sum', 'min', 'max', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate',
       'srate', 'drate', 'tnbpsrcip', 'tnbpdstip', 'tnp_psrcip', 'tnp_pdstip',
       'tnp_perproto', 'tnp_per_dport', 'ar_p_proto_p_srcip',
       'ar_p_proto_p_dstip', 'n_in_conn_p_srcip', 'n_in_conn_p_dstip',
       'ar_p_proto_p_sport', 'ar_p_proto_p_dport',
       'pkts_p_state_p_protocol_p_destip', 'pkts_p_state_p_protocol_p_srcip',
       'attack', 'category', 'subcategory ', 'saddr_192.168.100.149',
       'saddr_192.168.100.148', 'saddr_192.168.100.150',
       'saddr_192.168.100.147', 'saddr_192.168.100.3', 'saddr_192.168.100.7',
       'saddr_192.168.100.6', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'saddr_private', 'saddr_external',
       'daddr_192.168.100.149', 'daddr_192.168.100.148',
       'daddr_192.168.100.15

In [41]:
for flg in data.flgs.unique() :
    data[f"flgs_{flg}"]=data["flgs"].apply(lambda x : 1 if x==flg else 0)
for proto in data.proto.unique() :
    data[f"proto_{proto}"]=data["proto"].apply(lambda x : 1 if x==proto else 0)
for state in data.state.unique() :
    data[f"state_{state}"]=data["state"].apply(lambda x : 1 if x==state else 0)

In [42]:
data["attack_type"]=0
for i,type in enumerate(['Normal','DoS', 'DDoS', 'OS_Fingerprint', 'Service_Scan',
       'Keylogging', 'Data_Exfiltration']) :
    data["attack_type"]=data.apply(lambda x : (i if x["category"]==type else i if x["subcategory "]==type else x["attack_type"]),axis=1)

In [104]:
data.columns

Index(['pkSeqID', 'stime', 'flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport',
       'pkts', 'bytes', 'state', 'ltime', 'seq', 'dur', 'mean', 'stddev',
       'sum', 'min', 'max', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate',
       'srate', 'drate', 'tnbpsrcip', 'tnbpdstip', 'tnp_psrcip', 'tnp_pdstip',
       'tnp_perproto', 'tnp_per_dport', 'ar_p_proto_p_srcip',
       'ar_p_proto_p_dstip', 'n_in_conn_p_srcip', 'n_in_conn_p_dstip',
       'ar_p_proto_p_sport', 'ar_p_proto_p_dport',
       'pkts_p_state_p_protocol_p_destip', 'pkts_p_state_p_protocol_p_srcip',
       'attack', 'category', 'subcategory ', 'saddr_192.168.100.149',
       'saddr_192.168.100.148', 'saddr_192.168.100.150',
       'saddr_192.168.100.147', 'saddr_192.168.100.3', 'saddr_192.168.100.7',
       'saddr_192.168.100.6', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'saddr_private', 'saddr_external',
       'daddr_192.168.100.149', 'daddr_192.168.100.148',
       'daddr_192.168.100.15

In [44]:
data.columns

Index(['pkSeqID', 'stime', 'flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport',
       'pkts', 'bytes', 'state', 'ltime', 'seq', 'dur', 'mean', 'stddev',
       'sum', 'min', 'max', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate',
       'srate', 'drate', 'tnbpsrcip', 'tnbpdstip', 'tnp_psrcip', 'tnp_pdstip',
       'tnp_perproto', 'tnp_per_dport', 'ar_p_proto_p_srcip',
       'ar_p_proto_p_dstip', 'n_in_conn_p_srcip', 'n_in_conn_p_dstip',
       'ar_p_proto_p_sport', 'ar_p_proto_p_dport',
       'pkts_p_state_p_protocol_p_destip', 'pkts_p_state_p_protocol_p_srcip',
       'attack', 'category', 'subcategory ', 'saddr_192.168.100.149',
       'saddr_192.168.100.148', 'saddr_192.168.100.150',
       'saddr_192.168.100.147', 'saddr_192.168.100.3', 'saddr_192.168.100.7',
       'saddr_192.168.100.6', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'saddr_private', 'saddr_external',
       'daddr_192.168.100.149', 'daddr_192.168.100.148',
       'daddr_192.168.100.15

In [46]:
columns_wanted=['stime',  'flgs_e', 'flgs_e s', 'flgs_e dS', 'flgs_e g',
       'flgs_e *', 'flgs_eU', 'flgs_e &', 'flgs_e d', 'flgs_e    F',
       'flgs_e r', 'proto_tcp', 'proto_udp', 'proto_icmp',
       'proto_arp', 'proto_ipv6-icmp', 'proto_rarp', 'proto_igmp', 'saddr_192.168.100.149',
       'saddr_192.168.100.148', 'saddr_192.168.100.150',
       'saddr_192.168.100.147', 'saddr_192.168.100.3', 'saddr_192.168.100.7',
       'saddr_192.168.100.6', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'saddr_private', 'saddr_external','daddr_192.168.100.149', 'daddr_192.168.100.148',
       'daddr_192.168.100.150', 'daddr_192.168.100.147', 'daddr_192.168.100.3',
       'daddr_192.168.100.7', 'daddr_192.168.100.6', 'daddr_192.168.100.5',
       'daddr_192.168.100.4', 'daddr_192.168.100.1', 'daddr_private',
       'daddr_external','pkts', 'bytes', 'state_RST',
       'state_CON', 'state_INT', 'state_FIN', 'state_REQ', 'state_URP',
       'state_ECO', 'state_NRS', 'state_ACC', 'state_MAS', 'ltime', 'dur', 'mean', 'stddev',
       'sum', 'min', 'max', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate',
       'srate', 'drate', 'tnbpsrcip', 'tnbpdstip', 'tnp_psrcip', 'tnp_pdstip',
       'tnp_perproto', 'tnp_per_dport', 'ar_p_proto_p_srcip',
       'ar_p_proto_p_dstip', 'n_in_conn_p_srcip', 'n_in_conn_p_dstip',
       'ar_p_proto_p_sport', 'ar_p_proto_p_dport',
       'pkts_p_state_p_protocol_p_destip', 'pkts_p_state_p_protocol_p_srcip','attack_type']
data_sub=data[columns_wanted]

In [47]:
data_sub.head()

,stime,flgs_e,flgs_e s,flgs_e dS,flgs_e g,flgs_e *,flgs_eU,flgs_e &,flgs_e d,flgs_e F,...,tnp_per_dport,ar_p_proto_p_srcip,ar_p_proto_p_dstip,n_in_conn_p_srcip,n_in_conn_p_dstip,ar_p_proto_p_sport,ar_p_proto_p_dport,pkts_p_state_p_protocol_p_destip,pkts_p_state_p_protocol_p_srcip,attack_type
37247,1.526963e+09,1,0,0,0,0,0,0,0,0,...,8,4918.426175,939.965691,641.0,5208,564.567137,10430.133552,6482,23345,3
13201,1.528081e+09,0,1,0,0,0,0,0,0,0,...,13108094,3908.819243,1254.250649,120.0,6669,0.094425,100.862697,12994,15429,1
30334,1.526982e+09,1,0,0,0,0,0,0,0,0,...,21,3908.819243,1254.250649,120.0,6669,4368.510051,18097.726590,13229,23301,3
12636,1.528081e+09,0,1,0,0,0,0,0,0,0,...,13108094,4574.700595,9901.705447,146.0,5063,0.191534,100.862697,14111,30428,1
31374,1.526982e+09,1,0,0,0,0,0,0,0,0,...,23,4918.426175,1254.250649,641.0,6669,1856.201331,5937.703676,13229,23345,3


In [48]:
data_sub.to_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/Classifier_Data/bot_iot_dataset_preprocessed.csv",index=False)